In [ ]:
# ================================================================
#  CarbonSense AI — Full Web Dashboard
#  Paste ALL of this into ONE Google Colab cell and run it.
#  Uses Colab's FREE built-in tunnel — no ngrok, no token needed.
# ================================================================

# ── STEP 1: Install packages ──────────────────────────────────────
import subprocess, sys

for pkg in ["flask", "flask-cors", "langchain", "langchain-core",
            "langchain-nvidia-ai-endpoints", "langgraph"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Packages installed")


# ── STEP 2: PASTE YOUR NVIDIA API KEY HERE ────────────────────────
import os

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if NVIDIA_API_KEY is None:
    print("⚠️ Please set your NVIDIA_API_KEY environment variable")
else:
    print("✅ API key loaded")


# ── STEP 3: Agent + Tools (your original code) ───────────────────
import json, threading
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

llm = ChatNVIDIA(model="nvidia/nemotron-mini-4b-instruct", temperature=0.1)

# ── Data ──────────────────────────────────────────────────────────
GPU_WATTS = {
    "h100": 700, "a100": 400, "a10": 150, "rtx4090": 450,
    "v100": 300, "t4": 70, "l4": 72, "default": 400
}

GRID_CO2 = {
    "texas": 386, "california": 218, "washington": 118,
    "oregon": 185, "virginia": 320, "new york": 280,
    "germany": 380, "france": 85, "sweden": 45,
    "norway": 26,  "iceland": 28,  "canada": 150,
    "default": 350
}

WATER_FACTOR = 1.8  # liters per kWh

# ── Formulas ──────────────────────────────────────────────────────
def estimate_footprint(count, gpu, location, hours):
    """
    Carbon (kg CO2e) = GPU_count x GPU_power(kW) x hours x grid_intensity(kg/kWh)
    Water  (liters)  = energy(kWh) x water_usage_factor(L/kWh)
    """
    watts   = GPU_WATTS.get(gpu.lower(), GPU_WATTS["default"])
    co2_int = GRID_CO2.get(location.lower(), GRID_CO2["default"])
    energy  = count * (watts / 1000) * hours
    carbon  = energy * (co2_int / 1000)
    water   = energy * WATER_FACTOR
    trees   = carbon / 21.77
    car_km  = carbon / 0.21
    return {
        "count": count, "gpu": gpu, "location": location, "hours": hours,
        "watts": watts, "co2_int": co2_int,
        "energy_kwh":   round(energy, 1),
        "carbon_kg":    round(carbon, 2),
        "water_liters": round(water, 1),
        "trees":        round(trees, 1),
        "car_km":       round(car_km, 0),
    }

def parse_query(raw):
    prompt = f"""Extract GPU workload details from this text and return ONLY valid JSON.
Defaults if not mentioned: gpu="default", location="default", hours=24, count=1.
Map location to one of: {list(GRID_CO2.keys())}.
Map gpu to one of: {list(GPU_WATTS.keys())}.
Text: "{raw}"
Return JSON with exactly: {{"count": <int>, "gpu": "<str>", "location": "<str>", "hours": <int>}}"""
    resp = llm.invoke(prompt)
    text = resp.content.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

# ── Tools ─────────────────────────────────────────────────────────
@tool
def calculate_carbon(query: str) -> str:
    """Calculate carbon and water footprint for any GPU workload description."""
    p = parse_query(query)
    r = estimate_footprint(p["count"], p["gpu"], p["location"], p["hours"])
    return (
        f"Workload: {r['count']}x {r['gpu'].upper()} in {r['location'].title()} for {r['hours']}h\n"
        f"  GPU power      : {r['watts']}W per GPU\n"
        f"  Grid intensity : {r['co2_int']} gCO2/kWh\n"
        f"  Energy         : {r['energy_kwh']} kWh\n"
        f"  Carbon         : {r['carbon_kg']} kg CO2e\n"
        f"  Water          : {r['water_liters']} liters (at {WATER_FACTOR} L/kWh)\n"
        f"  Trees to offset: {r['trees']} trees/year\n"
        f"  Car equivalent : {r['car_km']:.0f} km driven"
    )

@tool
def suggest_alternatives(query: str) -> str:
    """Suggest greener regions and GPU practices for a workload."""
    p = parse_query(query)
    r = estimate_footprint(p["count"], p["gpu"], p["location"], p["hours"])
    current = r["co2_int"]
    lines = [
        f"Current region ({r['location'].title()}): {current} gCO2/kWh",
        f"Carbon: {r['carbon_kg']} kg CO2e | Water: {r['water_liters']} L\n",
        "Greener alternatives:"
    ]
    for region, intensity in sorted(GRID_CO2.items(), key=lambda x: x[1]):
        if region != "default" and intensity < current:
            alt     = estimate_footprint(r["count"], r["gpu"], region, r["hours"])
            saved   = round(r["carbon_kg"] - alt["carbon_kg"], 2)
            saved_w = round(r["water_liters"] - alt["water_liters"], 1)
            pct     = round((1 - intensity / current) * 100, 0)
            lines.append(
                f"  {region.title():<16} {intensity:>3} gCO2/kWh  "
                f"saves {saved} kg CO2 & {saved_w} L water ({pct:.0f}% less)"
            )
    lines += [
        "",
        "Operational tips:",
        "  1. Schedule training during off-peak/high-renewable hours (2-6 AM)",
        "  2. Use spot/preemptible instances to shift load to greener regions",
        "  3. Apply mixed-precision (FP16/BF16) to cut compute time ~30%",
        "  4. Consolidate low-utilisation clusters (<8% GPU util)",
        "  5. Purchase Renewable Energy Certificates (RECs) for residual emissions",
    ]
    return "\n".join(lines)

# ── LangGraph Agent ───────────────────────────────────────────────
tools = [calculate_carbon, suggest_alternatives]

AGENT_PROMPT = """You are CarbonSense AI, a sustainability advisor specialising in GPU infrastructure carbon and water footprints.
You help enterprises understand and reduce the environmental impact of their AI workloads.

You have access to these tools:
{tools}

Tool names: {tool_names}

ALWAYS follow this pattern:
Thought: <reasoning>
Action: <tool name>
Action Input: <input>
Observation: <result>
Thought: I now have enough information.
Final Answer: <full sustainability report>

In the Final Answer:
1. State carbon (kg CO2e), energy (kWh), and water (liters) clearly
2. Express in human terms: car trips (kg/0.21), trees/year (kg/21.77)
3. List top 3 greener region alternatives with exact savings
4. Give 3 operational tips specific to this workload
5. End with an encouraging bottom line

Question: {input}
{agent_scratchpad}"""

agent = create_react_agent(model=llm, tools=tools, prompt=AGENT_PROMPT)
print("✅ Agent ready")


# ── STEP 4: Flask server ──────────────────────────────────────────
from flask import Flask, request, jsonify, Response
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/analyze", methods=["POST"])
def analyze():
    try:
        data     = request.json or {}
        desc     = data.get("description", "").strip()
        gpu      = data.get("gpu", "a100")
        count    = int(data.get("count", 8))
        location = data.get("location", "virginia")
        hours    = int(data.get("hours", 24))

        estimates = estimate_footprint(count, gpu, location, hours)

        prompt = (
            f"{desc}\n\n" if desc else ""
        ) + (
            f"GPU: {gpu.upper()} x{count}, Region: {location}, Duration: {hours}h\n"
            f"Energy: {estimates['energy_kwh']} kWh | "
            f"Carbon: {estimates['carbon_kg']} kg CO2e | "
            f"Water: {estimates['water_liters']} L | "
            f"Trees: {estimates['trees']}/year"
        )

        result   = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
        analysis = result["messages"][-1].content

        return jsonify({"ok": True, "estimates": estimates, "analysis": analysis})

    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500


# ── STEP 5: Full HTML Dashboard ───────────────────────────────────
HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>CarbonSense AI</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap" rel="stylesheet"/>
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --g8:#0B5E3F;--g7:#0F6E56;--g6:#1D9E75;--g4:#5DCAA5;--g1:#E1F5EE;--g0:#f0faf6;
  --b7:#185FA5;--b1:#E6F1FB;--a7:#854F0B;--a4:#EF9F27;--a1:#FAEEDA;
  --r7:#A32D2D;--r1:#FCEBEB;
  --n9:#1a1a18;--n8:#2c2c2a;--n6:#5f5e5a;--n4:#888780;--n2:#d3d1c7;--n1:#f1efe8;--n0:#f8f7f3;
  --white:#fff;--bd:rgba(0,0,0,0.09);--bd2:rgba(0,0,0,0.14);
  --sh:0 1px 3px rgba(0,0,0,.06),0 1px 2px rgba(0,0,0,.04);
  --font:'DM Sans',sans-serif;--r:8px;--rl:12px;--rxl:16px;
}
html,body{height:100%;font-family:var(--font);background:var(--n0);color:var(--n9);font-size:14px;line-height:1.5}
.app{display:flex;height:100vh;overflow:hidden}

/* ── Sidebar ── */
.sb{width:218px;flex-shrink:0;background:var(--white);border-right:1px solid var(--bd);display:flex;flex-direction:column;overflow-y:auto}
.sb-logo{padding:18px 16px 14px;border-bottom:1px solid var(--bd);display:flex;align-items:center;gap:10px}
.sb-mark{width:32px;height:32px;background:var(--g8);border-radius:9px;display:flex;align-items:center;justify-content:center;flex-shrink:0}
.sb-mark svg{width:17px;height:17px}
.sb-name{font-size:14px;font-weight:600;color:var(--n9);letter-spacing:-.01em}
.sb-tier{font-size:10px;color:var(--n4)}
.nav-sec{padding:14px 10px 4px}
.nav-lbl{font-size:10px;font-weight:600;color:var(--n4);text-transform:uppercase;letter-spacing:.08em;padding:0 8px;margin-bottom:4px}
.nav-item{display:flex;align-items:center;gap:9px;padding:8px 10px;border-radius:var(--r);cursor:pointer;font-size:13px;color:var(--n6);transition:background .15s,color .15s}
.nav-item:hover{background:var(--n0);color:var(--n9)}
.nav-item.active{background:var(--g1);color:var(--g7);font-weight:500}
.nav-item svg{width:15px;height:15px;flex-shrink:0;opacity:.7}
.nav-item.active svg{opacity:1}
.sb-foot{margin-top:auto;padding:12px;border-top:1px solid var(--bd)}
.org-card{display:flex;align-items:center;gap:9px;padding:8px 10px;border-radius:var(--rl);border:1px solid var(--bd)}
.org-av{width:26px;height:26px;border-radius:6px;background:var(--b7);display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:600;color:#B5D4F4;flex-shrink:0}
.org-nm{font-size:12px;font-weight:500;color:var(--n9)}
.org-pl{font-size:10px;color:var(--n4)}

/* ── Main ── */
.main{flex:1;display:flex;flex-direction:column;overflow:hidden;min-width:0}
.topbar{padding:0 22px;height:54px;flex-shrink:0;background:var(--white);border-bottom:1px solid var(--bd);display:flex;align-items:center;justify-content:space-between;gap:16px}
.tb-title{font-size:15px;font-weight:600;color:var(--n9);letter-spacing:-.01em}
.tb-sub{font-size:11px;color:var(--n4);margin-top:1px}
.tb-r{display:flex;align-items:center;gap:8px;flex-shrink:0}
.pill{display:flex;align-items:center;gap:6px;padding:5px 10px;border-radius:20px;background:var(--n1);font-size:11px;color:var(--n6)}
.dot{width:6px;height:6px;border-radius:50%;background:var(--n4);flex-shrink:0}
.dot.ready{background:var(--g6);animation:blink 2s ease-in-out infinite}
.dot.busy{background:var(--a4);animation:blink .7s ease-in-out infinite}
@keyframes blink{0%,100%{opacity:1}50%{opacity:.35}}
.btn{padding:7px 13px;border-radius:var(--r);font-size:12px;font-weight:500;font-family:var(--font);cursor:pointer;border:1px solid var(--bd2);background:var(--white);color:var(--n8);transition:all .15s}
.btn:hover{background:var(--n0)}

/* ── Content ── */
.content{flex:1;overflow-y:auto;padding:18px 22px 32px}
.content::-webkit-scrollbar{width:5px}
.content::-webkit-scrollbar-thumb{background:var(--n2);border-radius:3px}

/* ── Input panel ── */
.inp-panel{background:var(--white);border:1px solid var(--bd);border-radius:var(--rxl);padding:18px 20px;margin-bottom:14px;box-shadow:var(--sh)}
.inp-title{font-size:14px;font-weight:600;color:var(--n9);margin-bottom:2px}
.inp-sub{font-size:12px;color:var(--n4);margin-bottom:12px}
.inp-desc{width:100%;padding:10px 12px;border:1px solid var(--bd2);border-radius:var(--r);font-family:var(--font);font-size:13px;color:var(--n9);background:var(--n0);resize:none;outline:none;line-height:1.6;margin-bottom:12px;transition:border-color .15s,box-shadow .15s}
.inp-desc:focus{border-color:var(--g6);background:var(--white);box-shadow:0 0 0 3px rgba(29,158,117,.1)}
.inp-desc::placeholder{color:var(--n4)}
.fields{display:grid;grid-template-columns:repeat(4,1fr) auto;gap:10px;align-items:end}
.fld{display:flex;flex-direction:column;gap:5px}
.fld-lbl{font-size:11px;font-weight:500;color:var(--n6)}
.fld select,.fld input{padding:8px 10px;border-radius:var(--r);border:1px solid var(--bd2);background:var(--n0);font-family:var(--font);font-size:12px;color:var(--n9);outline:none;width:100%;transition:border-color .15s}
.fld select:focus,.fld input:focus{border-color:var(--g6);background:var(--white);box-shadow:0 0 0 3px rgba(29,158,117,.1)}
.btn-go{padding:9px 20px;border-radius:var(--r);background:var(--g7);border:none;color:var(--white);font-family:var(--font);font-size:13px;font-weight:600;cursor:pointer;white-space:nowrap;height:38px;transition:background .15s,transform .1s;display:flex;align-items:center;gap:7px}
.btn-go:hover{background:var(--g8)}
.btn-go:active{transform:scale(.98)}
.btn-go:disabled{opacity:.5;cursor:not-allowed;transform:none}
.err-box{background:var(--r1);border:1px solid #F09595;border-radius:var(--r);padding:10px 14px;font-size:12px;color:var(--r7);margin-top:10px;display:none}

/* ── KPI ── */
.kpi-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:14px}
.kpi{background:var(--white);border:1px solid var(--bd);border-radius:var(--rxl);padding:15px 17px;box-shadow:var(--sh);position:relative;overflow:hidden}
.kpi-bar{position:absolute;top:0;left:0;right:0;height:3px;border-radius:var(--rxl) var(--rxl) 0 0}
.kpi-top{display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:9px}
.kpi-lbl{font-size:10px;font-weight:500;color:var(--n4);text-transform:uppercase;letter-spacing:.06em}
.kpi-ico{width:27px;height:27px;border-radius:7px;display:flex;align-items:center;justify-content:center;flex-shrink:0}
.kpi-ico svg{width:13px;height:13px}
.kpi-val{font-size:24px;font-weight:300;color:var(--n9);line-height:1;margin-bottom:4px;letter-spacing:-.02em}
.kpi-val span{font-size:12px;font-weight:400;color:var(--n4);margin-left:3px}
.kpi-sub{font-size:11px;color:var(--n4)}

/* ── Bottom ── */
.btm{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.card{background:var(--white);border:1px solid var(--bd);border-radius:var(--rxl);padding:18px 20px;box-shadow:var(--sh)}
.card-title{font-size:13px;font-weight:600;color:var(--n9);margin-bottom:2px}
.card-sub{font-size:11px;color:var(--n4);margin-bottom:14px}

/* ── Agent output ── */
.ph{display:flex;flex-direction:column;align-items:center;justify-content:center;padding:32px 20px;text-align:center}
.ph-ico{font-size:26px;margin-bottom:10px;opacity:.45}
.ph-txt{font-size:12px;color:var(--n4);line-height:1.6}
.thinking{display:flex;align-items:center;gap:10px;padding:24px 0}
.dots span{display:inline-block;width:7px;height:7px;border-radius:50%;background:var(--g6);animation:bounce 1.2s ease-in-out infinite;margin-right:4px}
.dots span:nth-child(2){animation-delay:.15s}
.dots span:nth-child(3){animation-delay:.3s}
@keyframes bounce{0%,80%,100%{transform:scale(.7);opacity:.3}40%{transform:scale(1);opacity:1}}
.think-lbl{font-size:12px;color:var(--n4)}
.result{font-size:13px;color:var(--n8);line-height:1.85;max-height:360px;overflow-y:auto;padding-right:4px}
.result::-webkit-scrollbar{width:4px}
.result::-webkit-scrollbar-thumb{background:var(--n2);border-radius:2px}
.result h3{font-size:13px;font-weight:600;color:var(--n9);margin:14px 0 5px}
.result h3:first-child{margin-top:0}
.result p{margin-bottom:7px}
.result ul{padding-left:16px;margin-bottom:7px}
.result li{margin-bottom:3px}
.result strong{font-weight:600;color:var(--n9)}

/* ── Region table ── */
.rtbl{width:100%;border-collapse:collapse}
.rtbl th{font-size:10px;font-weight:600;color:var(--n4);text-transform:uppercase;letter-spacing:.07em;padding:0 0 9px;text-align:left;border-bottom:1px solid var(--n1)}
.rtbl td{font-size:12px;color:var(--n8);padding:7px 0;border-bottom:1px solid var(--n1);vertical-align:middle}
.rtbl tr:last-child td{border-bottom:none}
.rtbl tr.hi td{background:var(--g0)}
.rbar-bg{width:70px;height:5px;background:var(--n1);border-radius:3px;display:inline-block;vertical-align:middle}
.rbar{height:5px;border-radius:3px;display:inline-block}
.rbadge{font-size:9px;font-weight:600;padding:2px 6px;border-radius:10px}

@keyframes fadeUp{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:translateY(0)}}
.fade{animation:fadeUp .3s ease forwards}
@keyframes spin{to{transform:rotate(360deg)}}

@media(max-width:1100px){.fields{grid-template-columns:1fr 1fr}.kpi-grid{grid-template-columns:1fr 1fr}.btm{grid-template-columns:1fr}}
@media(max-width:768px){.sb{display:none}}
</style>
</head>
<body>
<div class="app">

  <!-- Sidebar -->
  <aside class="sb">
    <div class="sb-logo">
      <div class="sb-mark">
        <svg viewBox="0 0 18 18" fill="none">
          <path d="M9 2C6.5 2 5 3.8 5 5.8C5 7.5 6 9 7 9.8L7 13H11L11 9.8C12 9 13 7.5 13 5.8C13 3.8 11.5 2 9 2Z" fill="#9FE1CB"/>
          <path d="M7 13H11V14.5C11 15.1 10.4 15.5 9.5 15.5H8.5C7.6 15.5 7 15.1 7 14.5V13Z" fill="#5DCAA5"/>
          <line x1="3" y1="7.2" x2="5" y2="6.5" stroke="#5DCAA5" stroke-width="1.2" stroke-linecap="round"/>
          <line x1="15" y1="7.2" x2="13" y2="6.5" stroke="#5DCAA5" stroke-width="1.2" stroke-linecap="round"/>
          <line x1="3.5" y1="10.5" x2="5.5" y2="9.5" stroke="#5DCAA5" stroke-width="1.2" stroke-linecap="round"/>
          <line x1="14.5" y1="10.5" x2="12.5" y2="9.5" stroke="#5DCAA5" stroke-width="1.2" stroke-linecap="round"/>
        </svg>
      </div>
      <div>
        <div class="sb-name">CarbonSense AI</div>
        <div class="sb-tier">Enterprise Platform</div>
      </div>
    </div>
    <div class="nav-sec">
      <div class="nav-lbl">Overview</div>
      <div class="nav-item active"><svg viewBox="0 0 16 16" fill="none"><rect x="2" y="2" width="5" height="5" rx="1.5" fill="currentColor"/><rect x="9" y="2" width="5" height="5" rx="1.5" fill="currentColor" opacity=".5"/><rect x="2" y="9" width="5" height="5" rx="1.5" fill="currentColor" opacity=".5"/><rect x="9" y="9" width="5" height="5" rx="1.5" fill="currentColor" opacity=".5"/></svg>Dashboard</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><rect x="2" y="4" width="12" height="8" rx="2" stroke="currentColor" stroke-width="1.2" fill="none"/><path d="M5 4V3M8 4V3M11 4V3" stroke="currentColor" stroke-width="1.2" stroke-linecap="round"/></svg>AI Workloads</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><rect x="2" y="6" width="12" height="8" rx="1.5" stroke="currentColor" stroke-width="1.2" fill="none"/><path d="M5 6V4C5 2.9 6 2 8 2C10 2 11 2.9 11 4V6" stroke="currentColor" stroke-width="1.2" fill="none"/></svg>Data Centers</div>
    </div>
    <div class="nav-sec">
      <div class="nav-lbl">Emissions</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><circle cx="8" cy="8" r="5.5" stroke="currentColor" stroke-width="1.2" fill="none"/><path d="M8 5V8L10 10" stroke="currentColor" stroke-width="1.2" stroke-linecap="round"/></svg>Carbon Report</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><path d="M2 13L5 9L8 11L12 5L14 7" stroke="currentColor" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" fill="none"/></svg>Scope 1 / 2 / 3</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><rect x="3" y="2" width="10" height="12" rx="2" stroke="currentColor" stroke-width="1.2" fill="none"/><line x1="5.5" y1="6" x2="10.5" y2="6" stroke="currentColor" stroke-width="1.2" stroke-linecap="round"/><line x1="5.5" y1="9" x2="10.5" y2="9" stroke="currentColor" stroke-width="1.2" stroke-linecap="round"/></svg>Offsets &amp; Credits</div>
    </div>
    <div class="nav-sec">
      <div class="nav-lbl">Resources</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><path d="M8 2L4 7.5C4 10.5 5.8 13 8 13C10.2 13 12 10.5 12 7.5L8 2Z" stroke="currentColor" stroke-width="1.2" fill="none"/></svg>Water Usage</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><path d="M5 8L8 3L11 8L8 13L5 8Z" stroke="currentColor" stroke-width="1.2" fill="none"/></svg>GPU Efficiency</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><circle cx="8" cy="8" r="3" stroke="currentColor" stroke-width="1.2" fill="none"/><path d="M8 2V4M8 12V14M2 8H4M12 8H14" stroke="currentColor" stroke-width="1.2" stroke-linecap="round"/></svg>Energy Mix</div>
    </div>
    <div class="nav-sec">
      <div class="nav-lbl">Compliance</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><rect x="3" y="2" width="10" height="12" rx="2" stroke="currentColor" stroke-width="1.2" fill="none"/><line x1="5.5" y1="6" x2="10.5" y2="6" stroke="currentColor" stroke-width="1.2" stroke-linecap="round"/></svg>ESG Reports</div>
      <div class="nav-item"><svg viewBox="0 0 16 16" fill="none"><path d="M2 4H14M4 4V3C4 2.4 4.4 2 5 2H11C11.6 2 12 2.4 12 3V4" stroke="currentColor" stroke-width="1.2" stroke-linecap="round" fill="none"/><rect x="2" y="4" width="12" height="10" rx="1.5" stroke="currentColor" stroke-width="1.2" fill="none"/></svg>Audit Trail</div>
    </div>
    <div class="sb-foot">
      <div class="org-card">
        <div class="org-av">NV</div>
        <div><div class="org-nm">NVIDIA Corp</div><div class="org-pl">Enterprise · 2025</div></div>
      </div>
    </div>
  </aside>

  <!-- Main -->
  <div class="main">
    <header class="topbar">
      <div>
        <div class="tb-title">Workload Carbon Analyzer</div>
        <div class="tb-sub" id="tbSub">Fill in your workload details and click Analyze</div>
      </div>
      <div class="tb-r">
        <div class="pill"><div class="dot ready" id="dot"></div><span id="stTxt">Ready</span></div>
        <button class="btn" onclick="exportReport()">Export Report</button>
      </div>
    </header>

    <div class="content">

      <!-- Input panel -->
      <div class="inp-panel">
        <div class="inp-title">Describe your AI workload</div>
        <div class="inp-sub">Enter a natural language description, or fill in the fields — the agent uses both to reason through your footprint</div>
        <textarea class="inp-desc" id="desc" rows="2"
          placeholder="e.g. Fine-tuning a 70B LLaMA model using 64 H100 GPUs in Virginia for 72 hours"></textarea>
        <div class="fields">
          <div class="fld">
            <label class="fld-lbl">GPU type</label>
            <select id="gpu">
              <option value="a100">A100 — 400W</option>
              <option value="h100">H100 — 700W</option>
              <option value="a10">A10 — 150W</option>
              <option value="v100">V100 — 300W</option>
              <option value="t4">T4 — 70W</option>
              <option value="l4">L4 — 72W</option>
              <option value="rtx4090">RTX 4090 — 450W</option>
            </select>
          </div>
          <div class="fld">
            <label class="fld-lbl">Number of GPUs</label>
            <input type="number" id="count" value="8" min="1" placeholder="e.g. 64"/>
          </div>
          <div class="fld">
            <label class="fld-lbl">Region</label>
            <select id="region">
              <option value="virginia">Virginia — 320 gCO₂/kWh</option>
              <option value="texas">Texas — 386 gCO₂/kWh</option>
              <option value="california">California — 218 gCO₂/kWh</option>
              <option value="washington">Washington — 118 gCO₂/kWh</option>
              <option value="oregon">Oregon — 185 gCO₂/kWh</option>
              <option value="new york">New York — 280 gCO₂/kWh</option>
              <option value="germany">Germany — 380 gCO₂/kWh</option>
              <option value="france">France — 85 gCO₂/kWh</option>
              <option value="sweden">Sweden — 45 gCO₂/kWh</option>
              <option value="norway">Norway — 26 gCO₂/kWh</option>
              <option value="iceland">Iceland — 28 gCO₂/kWh</option>
              <option value="canada">Canada — 150 gCO₂/kWh</option>
            </select>
          </div>
          <div class="fld">
            <label class="fld-lbl">Duration (hours)</label>
            <input type="number" id="hours" value="24" min="1" placeholder="e.g. 72"/>
          </div>
          <button class="btn-go" id="goBtn" onclick="runAnalysis()">
            <svg width="13" height="13" viewBox="0 0 14 14" fill="none"><path d="M3 7H11M8 4L11 7L8 10" stroke="white" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></svg>
            Analyze
          </button>
        </div>
        <div class="err-box" id="errBox"></div>
      </div>

      <!-- KPI cards -->
      <div class="kpi-grid">
        <div class="kpi">
          <div class="kpi-bar" style="background:#EF9F27"></div>
          <div class="kpi-top">
            <div class="kpi-lbl">Energy Used</div>
            <div class="kpi-ico" style="background:#FAEEDA"><svg viewBox="0 0 14 14" fill="none"><path d="M4.5 7L7 2L9.5 7L7 12L4.5 7Z" stroke="#854F0B" stroke-width="1.2" fill="none"/></svg></div>
          </div>
          <div class="kpi-val" id="kE">—<span>kWh</span></div>
          <div class="kpi-sub">Total cluster energy</div>
        </div>
        <div class="kpi">
          <div class="kpi-bar" style="background:#1D9E75"></div>
          <div class="kpi-top">
            <div class="kpi-lbl">Carbon Emissions</div>
            <div class="kpi-ico" style="background:#E1F5EE"><svg viewBox="0 0 14 14" fill="none"><circle cx="7" cy="7" r="4.5" stroke="#0F6E56" stroke-width="1.2" fill="none"/><path d="M7 4V7L9 9" stroke="#0F6E56" stroke-width="1.2" stroke-linecap="round"/></svg></div>
          </div>
          <div class="kpi-val" id="kC">—<span>kg CO₂e</span></div>
          <div class="kpi-sub">Total footprint</div>
        </div>
        <div class="kpi">
          <div class="kpi-bar" style="background:#185FA5"></div>
          <div class="kpi-top">
            <div class="kpi-lbl">Water Usage</div>
            <div class="kpi-ico" style="background:#E6F1FB"><svg viewBox="0 0 14 14" fill="none"><path d="M7 2L3.5 7C3.5 9.5 5 11 7 11C9 11 10.5 9.5 10.5 7L7 2Z" stroke="#185FA5" stroke-width="1.2" fill="none"/></svg></div>
          </div>
          <div class="kpi-val" id="kW">—<span>liters</span></div>
          <div class="kpi-sub">Cooling consumption</div>
        </div>
        <div class="kpi">
          <div class="kpi-bar" style="background:#639922"></div>
          <div class="kpi-top">
            <div class="kpi-lbl">Trees to Offset</div>
            <div class="kpi-ico" style="background:#EAF3DE"><svg viewBox="0 0 14 14" fill="none"><polyline points="2,11 5,7 7.5,8.5 12,3" stroke="#3B6D11" stroke-width="1.2" fill="none" stroke-linecap="round" stroke-linejoin="round"/></svg></div>
          </div>
          <div class="kpi-val" id="kT">—<span>trees/yr</span></div>
          <div class="kpi-sub">Annual offset needed</div>
        </div>
      </div>

      <!-- Bottom -->
      <div class="btm">
        <div class="card">
          <div class="card-title">CarbonSense agent analysis</div>
          <div class="card-sub" id="agSub">NVIDIA Nemotron · AI reasoning + sustainability recommendations</div>
          <div id="agOut">
            <div class="ph">
              <div class="ph-ico">🌱</div>
              <div class="ph-txt">Fill in your workload details above and click <strong>Analyze</strong>.<br/>The agent will calculate your carbon &amp; water footprint and suggest greener alternatives.</div>
            </div>
          </div>
        </div>
        <div class="card">
          <div class="card-title">Region carbon intensity</div>
          <div class="card-sub" id="regSub">All regions — sorted by grid intensity</div>
          <table class="rtbl">
            <thead><tr><th>Region</th><th>gCO₂/kWh</th><th style="width:78px">Level</th><th>Rating</th></tr></thead>
            <tbody id="regBody"></tbody>
          </table>
        </div>
      </div>

    </div><!-- /content -->
  </div><!-- /main -->
</div><!-- /app -->

<script>
const REGIONS=[
  {n:"Norway",v:26,c:"#1D9E75"},{n:"Iceland",v:28,c:"#1D9E75"},
  {n:"Sweden",v:45,c:"#5DCAA5"},{n:"France",v:85,c:"#5DCAA5"},
  {n:"Washington",v:118,c:"#9FE1CB"},{n:"Canada",v:150,c:"#EF9F27"},
  {n:"Oregon",v:185,c:"#EF9F27"},{n:"California",v:218,c:"#EF9F27"},
  {n:"New York",v:280,c:"#D85A30"},{n:"Virginia",v:320,c:"#D85A30"},
  {n:"Germany",v:380,c:"#E24B4A"},{n:"Texas",v:386,c:"#E24B4A"},
];
function rating(v){
  if(v<=50)  return["Best","#dcfce7","#166534"];
  if(v<=150) return["Low","#d1fae5","#065f46"];
  if(v<=250) return["Med","#fef3c7","#92400e"];
  return["High","#fee2e2","#991b1b"];
}
function buildTable(active){
  const b=document.getElementById("regBody"); b.innerHTML="";
  REGIONS.forEach(r=>{
    const[lbl,bg,fg]=rating(r.v);
    const w=Math.round(r.v/400*100);
    const on=active&&r.n.toLowerCase()===active.toLowerCase();
    const tr=document.createElement("tr");
    if(on)tr.className="hi";
    tr.innerHTML=`<td style="font-weight:${on?600:400}">${on?"→ ":""}${r.n}</td>
      <td>${r.v}</td>
      <td><div class="rbar-bg"><div class="rbar" style="width:${w}%;background:${r.c}"></div></div></td>
      <td><span class="rbadge" style="background:${bg};color:${fg}">${lbl}</span></td>`;
    b.appendChild(tr);
  });
}
function setStatus(s,t){
  document.getElementById("dot").className="dot "+s;
  document.getElementById("stTxt").textContent=t;
  document.getElementById("tbSub").textContent=t;
}
function fmt(n){return Number(n).toLocaleString()}
function renderResult(text){
  let h=text
    .replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;")
    .replace(/^#{1,3} (.+)$/gm,"<h3>$1</h3>")
    .replace(/\*\*(.+?)\*\*/g,"<strong>$1</strong>")
    .replace(/^[\-•\*] (.+)$/gm,"<li>$1</li>")
    .replace(/(<li>[\s\S]*?<\/li>)/g,"<ul>$1</ul>")
    .replace(/\n\n+/g,"</p><p>").replace(/\n/g,"<br/>");
  return`<div class="result fade"><p>${h}</p></div>`;
}
async function runAnalysis(){
  const desc=document.getElementById("desc").value.trim();
  const gpu=document.getElementById("gpu").value;
  const count=parseInt(document.getElementById("count").value)||8;
  const region=document.getElementById("region").value;
  const hours=parseInt(document.getElementById("hours").value)||24;
  document.getElementById("errBox").style.display="none";
  const btn=document.getElementById("goBtn");
  btn.disabled=true;
  btn.innerHTML=`<svg width="13" height="13" viewBox="0 0 14 14" fill="none" style="animation:spin 1s linear infinite"><path d="M7 1.5A5.5 5.5 0 1 1 1.5 7" stroke="white" stroke-width="1.5" stroke-linecap="round" fill="none"/></svg> Analyzing…`;
  setStatus("busy","Analyzing with NVIDIA Nemotron…");
  buildTable(region);
  document.getElementById("agOut").innerHTML=`<div class="thinking"><div class="dots"><span></span><span></span><span></span></div><div class="think-lbl">CarbonSense AI is reasoning through your workload…</div></div>`;
  document.getElementById("agSub").textContent="NVIDIA Nemotron · processing…";
  try{
    const res=await fetch("/analyze",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({description:desc,gpu,count,location:region,hours})});
    const data=await res.json();
    if(!data.ok)throw new Error(data.error||"Unknown error");
    const e=data.estimates;
    document.getElementById("kE").innerHTML=`${fmt(e.energy_kwh)}<span>kWh</span>`;
    document.getElementById("kC").innerHTML=`${fmt(e.carbon_kg)}<span>kg CO₂e</span>`;
    document.getElementById("kW").innerHTML=`${fmt(e.water_liters)}<span>liters</span>`;
    document.getElementById("kT").innerHTML=`${fmt(e.trees)}<span>trees/yr</span>`;
    document.getElementById("agOut").innerHTML=renderResult(data.analysis);
    document.getElementById("agSub").textContent=`Analysis complete · ${new Date().toLocaleTimeString()}`;
    document.getElementById("regSub").textContent=`${region.charAt(0).toUpperCase()+region.slice(1)} · ${e.co2_int} gCO₂/kWh`;
    buildTable(region);
    setStatus("ready",`Done · ${count}× ${gpu.toUpperCase()} · ${region}`);
  }catch(ex){
    document.getElementById("errBox").textContent="Error: "+ex.message;
    document.getElementById("errBox").style.display="block";
    document.getElementById("agOut").innerHTML=`<div class="ph"><div class="ph-ico">⚠️</div><div class="ph-txt">${ex.message}</div></div>`;
    setStatus("ready","Error — see message below");
  }
  btn.disabled=false;
  btn.innerHTML=`<svg width="13" height="13" viewBox="0 0 14 14" fill="none"><path d="M3 7H11M8 4L11 7L8 10" stroke="white" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></svg> Analyze`;
}
function exportReport(){
  const lines=["CarbonSense AI — ESG Report","=".repeat(48),"Generated: "+new Date().toLocaleString(),"","WORKLOAD","—".repeat(30),"Description : "+(document.getElementById("desc").value||"N/A"),"GPU         : "+document.getElementById("gpu").value.toUpperCase(),"Count       : "+document.getElementById("count").value,"Region      : "+document.getElementById("region").value,"Hours       : "+document.getElementById("hours").value,"","ESTIMATES","—".repeat(30),"Energy  : "+document.getElementById("kE").innerText,"Carbon  : "+document.getElementById("kC").innerText,"Water   : "+document.getElementById("kW").innerText,"Offset  : "+document.getElementById("kT").innerText,"","AGENT ANALYSIS","—".repeat(30),document.getElementById("agOut").innerText,"","Powered by CarbonSense AI · NVIDIA Nemotron"];
  const a=document.createElement("a");
  a.href=URL.createObjectURL(new Blob([lines.join("\n")],{type:"text/plain"}));
  a.download="carbonsense_report.txt"; a.click();
}
buildTable();
</script>
</body>
</html>"""

@app.route("/")
def index():
    return Response(HTML, mimetype="text/html")


# ── STEP 6: Launch with Colab's built-in tunnel (FREE, no token) ─
import time
from google.colab import output
from google.colab.output import serve_kernel_port_as_iframe

# Start Flask in background thread
flask_thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5000, use_reloader=False, debug=False)
)
flask_thread.daemon = True
flask_thread.start()
time.sleep(2)

print("\n" + "="*55)
print("  ✅  CarbonSense AI is running!")
print("  🌐  The dashboard will appear below this cell")
print("="*55)
print("\n  Fill in GPU type, count, region, hours → Analyze")
print("  The NVIDIA Nemotron agent calculates your footprint\n")

# Serve the dashboard inline inside Colab output
output.serve_kernel_port_as_iframe(5000, height="800")

✅ Packages installed
✅ API key set
✅ Agent ready
 * Serving Flask app '__main__'
 * Debug mode: off


/tmp/ipykernel_24128/2979487774.py:166: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools, prompt=AGENT_PROMPT)
/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/chat_models.py:1091: UserWarning: Model 'nvidia/nemotron-mini-4b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



  ✅  CarbonSense AI is running!
  🌐  The dashboard will appear below this cell

  Fill in GPU type, count, region, hours → Analyze
  The NVIDIA Nemotron agent calculates your footprint



<IPython.core.display.Javascript object>